In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. CONFIGURATION & DATA LOADING
# ==========================================
print("=== STARTING LATENT PROCESS MIXED MODEL (LPMM) BASELINE ===")

DATA_FILE = "proact_preprocessed_S1.csv"  # UPDATE THIS

df = pd.read_csv(DATA_FILE)

all_items = [
    'Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing', 
    'Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene', 
    'Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs', 
    'R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency'
]

meas_covariates = ['Sex_Female', 'Treatment_Active', 'Age_Base']
dyn_covariates = ['Bulbar_Onset', 'FVC_Base', 'BMI_Base']

# Drop NAs for clean modeling
df_clean = df.dropna(subset=all_items + meas_covariates + dyn_covariates + ['ALSFRS_Delta']).copy()
df_clean['Total_Score'] = df_clean[all_items].sum(axis=1)

# Standardize the Total Score to act as the proxy for the continuous Latent Process Lambda(t)
df_clean['Total_Z'] = (df_clean['Total_Score'] - df_clean['Total_Score'].mean()) / df_clean['Total_Score'].std()

# ==========================================
# 1. FIT THE DYNAMIC LATENT TRAJECTORY
# ==========================================
print("Fitting Continuous Latent Trajectory (Dynamic Covariates)...")

# Equation (1) from Proust-Lima: Modeling the latent process trajectory
dyn_formula = "Total_Z ~ ALSFRS_Delta + " + " + ".join(dyn_covariates)
latent_model = smf.mixedlm(dyn_formula, data=df_clean, groups=df_clean["subject_id"]).fit(reml=False, disp=False)

# Save the estimated latent trajectory Lambda_i(t) for every visit
df_clean['Lambda_it'] = latent_model.fittedvalues

# Extract Global Dynamic Z-Scores
# Note: In a true LPMM, dynamic covariates affect the global trait, not individual items.
dyn_z_scores = {cov: latent_model.tvalues[cov] for cov in dyn_covariates}

# ==========================================
# 2. FIT ITEM-SPECIFIC MEASUREMENT MODELS
# ==========================================
print("Fitting Non-Linear Measurement Models (IRT Threshold Links)...")

metrics_results = []
covariate_results = []
expected_scores_matrix = np.zeros((len(df_clean), len(all_items)))
actual_scores_matrix = df_clean[all_items].values

for k, item in enumerate(all_items):
    df_clean['Item_Cat'] = pd.Categorical(df_clean[item].astype(int), ordered=True)
    
    # Equation (2) from Proust-Lima: Ordered Logit mapping Lambda(t) + measurement covariates to ordinal bounds
    exog_vars = ['Lambda_it'] + meas_covariates
    try:
        meas_model = OrderedModel(df_clean['Item_Cat'], df_clean[exog_vars], distr='logit').fit(method='bfgs', disp=False)
        
        # 1. Extract Measurement Covariate Z-scores
        for cov in meas_covariates:
            covariate_results.append({
                'Item': item, 'Covariate': cov, 
                'Model': '5. LPMM (Proust-Lima Approx)',
                'Z_Score': meas_model.tvalues[cov]
            })
            
        # 2. Add the Global Dynamic Covariate Z-scores (Mapped to all items for matrix compatibility)
        for cov in dyn_covariates:
            covariate_results.append({
                'Item': item, 'Covariate': cov, 
                'Model': '5. LPMM (Proust-Lima Approx)',
                'Z_Score': dyn_z_scores[cov]
            })
            
        # 3. Calculate Expected Scores for Predictive Metrics
        # predict() returns probabilities for categories 0, 1, 2, 3, 4
        probs = meas_model.predict(df_clean[exog_vars])
        expected_score = (probs.values * np.arange(5)).sum(axis=1)
        expected_scores_matrix[:, k] = expected_score
        
    except Exception as e:
        print(f"Failed to fit item {item}: {e}")

# ==========================================
# 3. CALCULATE PREDICTIVE METRICS
# ==========================================
print("Calculating Predictive Accuracy Metrics...\n")

for k, item in enumerate(all_items):
    actual = actual_scores_matrix[:, k]
    expected = expected_scores_matrix[:, k]
    
    rmse = np.sqrt(mean_squared_error(actual, expected))
    mae = mean_absolute_error(actual, expected)
    pred_cat = np.clip(np.round(expected), 0, 4)
    exact = np.mean(pred_cat == actual) * 100
    kappa = cohen_kappa_score(actual, pred_cat, weights='linear')
    
    metrics_results.append({
        'Level': 'Item', 'Target': item, 
        'RMSE': rmse, 'MAE': mae, 'Exact_%': exact, 'Kappa': kappa
    })

# Add Total Score
actual_total = actual_scores_matrix.sum(axis=1)
expected_total = expected_scores_matrix.sum(axis=1)
metrics_results.append({
    'Level': 'Overall', 'Target': 'Total Score',
    'RMSE': np.sqrt(mean_squared_error(actual_total, expected_total)),
    'MAE': mean_absolute_error(actual_total, expected_total),
    'Exact_%': np.nan, 'Kappa': np.nan
})

# ==========================================
# 4. CONSOLE OUTPUT
# ==========================================
print("="*85)
print("BASELINE 5: LATENT PROCESS MIXED MODEL (LPMM) METRICS")
print("="*85)
df_metrics = pd.DataFrame(metrics_results).set_index(['Level', 'Target'])
print(df_metrics.to_string(float_format="{:.3f}".format, na_rep="-"))

print("\n" + "="*85)
print("TERMINAL EXPORT FOR COVARIATE Z-SCORES (LPMM BASELINE)")
print("="*85)

df_covs = pd.DataFrame(covariate_results)
for cov in meas_covariates + dyn_covariates:
    print(f"\n---> COVARIATE: {cov.upper()} <---")
    subset = df_covs[df_covs['Covariate'] == cov]
    if not subset.empty:
        pivot_z = subset.pivot(index='Item', columns='Model', values='Z_Score')
        print(pivot_z.to_string(float_format="{:.3f}".format))

=== STARTING LATENT PROCESS MIXED MODEL (LPMM) BASELINE ===
Fitting Continuous Latent Trajectory (Dynamic Covariates)...
Fitting Non-Linear Measurement Models (IRT Threshold Links)...
Calculating Predictive Accuracy Metrics...

BASELINE 5: LATENT PROCESS MIXED MODEL (LPMM) METRICS
                                       RMSE   MAE  Exact_%  Kappa
Level   Target                                                   
Item    Q1_Speech                     0.997 0.776   40.221  0.220
        Q2_Salivation                 0.872 0.667   49.400  0.184
        Q3_Swallowing                 0.772 0.575   57.242  0.299
        Q4_Handwriting                0.862 0.664   47.417  0.349
        Q5_Cutting                    0.910 0.743   38.123  0.410
        Q6_Dressing_and_Hygiene       0.770 0.622   46.840  0.450
        Q7_Turning_in_Bed             0.760 0.597   52.191  0.504
        Q8_Walking                    0.838 0.685   41.052  0.280
        Q9_Climbing_Stairs            1.110 0.922   30.051